In [ ]:
# Shared Setup: run this cell first before any section
# Committed by: [Repo Owner Name]

import pandas as pd

# -----------------------------------------------
# Dataset: Q1 2024 Sales Records
# 20 orders — date, region, product, quantity, price, sales rep
# -----------------------------------------------

sales_raw = {
    'OrderID':   [1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,
                  1011,1012,1013,1014,1015,1016,1017,1018,1019,1020],
    'Date':      ['2024-01-05','2024-01-12','2024-01-18','2024-01-22','2024-01-30',
                  '2024-02-03','2024-02-08','2024-02-14','2024-02-19','2024-02-25',
                  '2024-03-02','2024-03-07','2024-03-11','2024-03-15','2024-03-18',
                  '2024-03-20','2024-03-22','2024-03-25','2024-03-27','2024-03-30'],
    'Region':    ['North','South','East','West','North','East','South','North','West','East',
                  'South','North','East','West','South','North','East','West','South','North'],
    'Product':   ['Laptop','Headset','Monitor','Keyboard','Laptop','Webcam','Headset','Monitor',
                  'Laptop','Keyboard','Webcam','Laptop','Monitor','Headset','Keyboard','Webcam',
                  'Laptop','Monitor','Headset','Keyboard'],
    'Quantity':  [2,5,1,8,1,10,3,2,1,12,7,2,1,4,6,9,1,2,5,10],
    'UnitPrice': [1200.00,89.99,499.00,49.99,1200.00,79.99,89.99,499.00,1200.00,49.99,
                  79.99,1200.00,499.00,89.99,49.99,79.99,1200.00,499.00,89.99,49.99],
    'SalesRep':  ['Jordan','Taylor','Morgan','Jordan','Casey','Taylor','Morgan','Casey',
                  'Jordan','Taylor','Casey','Morgan','Jordan','Taylor','Casey','Morgan',
                  'Jordan','Casey','Taylor','Morgan']
}

df = pd.DataFrame(sales_raw)
df['Date'] = pd.to_datetime(df['Date'])
df['Revenue'] = df['Quantity'] * df['UnitPrice']

print('Dataset loaded successfully.')
print(f'  Orders: {len(df)}')
print(f'  Date range: {df.Date.min().date()} to {df.Date.max().date()}')
print(f'  Total revenue: ${df.Revenue.sum():,.2f}')

In [ ]:
# Section 1, Cell 1: Explore the dataset
# Run the Shared Setup cell above first

# TODO: Display basic info about the dataset
# Use: df.shape, df.dtypes, df.head(), df.describe()

print('Dataset shape:', df.shape)
print()
print('Column types:')
print(df.dtypes)
print()
print('First 5 rows:')
print(df.head())

# Clear outputs -> Save -> Commit: 'Section 1: Add initial exploration - [Name]'

In [ ]:
# Section 1, Cell 2: Data quality check

def check_data_quality(dataframe):
    """Check the dataset for missing values, duplicates, and out-of-range values.

    Args:
        dataframe: the sales DataFrame

    Returns:
        dict with keys: 'missing_values', 'duplicate_orders', 'negative_quantities'
        Each value is an integer count.

    >>> import pandas as pd
    >>> test_df = pd.DataFrame({'OrderID': [1,2,2], 'Quantity': [5, -1, 3],
    ...                         'Revenue': [100, 50, 75]})
    >>> result = check_data_quality(test_df)
    >>> result['duplicate_orders']
    2
    >>> result['negative_quantities']
    1
    """
    # TODO: Use Copilot to implement this
    # Hint: df.isnull().sum().sum(), df.duplicated(), df[df['Quantity'] < 0]
    missing_values = dataframe.isnull().sum().sum()
    duplicate_orders = dataframe.duplicated(subset='OrderID').sum()
    negative_quantities = dataframe[dataframe['Quantity'] < 0].shape[0]

    return {
        'missing_values': missing_values,
        'duplicate_orders': duplicate_orders,
        'negative_quantities': negative_quantities
    }


# Test it
quality = check_data_quality(df)
print('Data quality report:')
for key, value in quality.items():
    status = 'OK' if value == 0 else f'WARNING: {value} found'
    print(f'  {key}: {status}')

# Clear outputs -> Commit: 'Section 1: Add check_data_quality() - [Name]'

In [ ]:
# Section 1, Cell 3: Unique value counts

def dataset_summary(dataframe):
    """Summarize unique values and ranges in the dataset.

    Args:
        dataframe: the sales DataFrame

    Returns:
        dict with keys:
          'num_products': int, count of unique products
          'num_regions': int, count of unique regions
          'num_reps': int, count of unique sales reps
          'revenue_range': tuple (min_revenue, max_revenue) per order

    >>> import pandas as pd
    >>> test_df = pd.DataFrame({'Product': ['A','B','A'], 'Region': ['N','S','N'],
    ...                         'SalesRep': ['X','X','Y'], 'Revenue': [100, 50, 200]})
    >>> s = dataset_summary(test_df)
    >>> s['num_products']
    2
    >>> s['num_reps']
    2
    """
    num_products = dataframe['Product'].nunique()
    num_regions = dataframe['Region'].nunique()
    num_reps = dataframe['SalesRep'].nunique()
    revenue_range = (dataframe['Revenue'].min(), dataframe['Revenue'].max())

    return {
        'num_products': num_products,
        'num_regions': num_regions,
        'num_reps': num_reps,
        'revenue_range': revenue_range
    }


summary = dataset_summary(df)
print('Dataset summary:')
for key, value in summary.items():
    print(f'  {key}: {value}')

# Clear outputs -> Commit: 'Section 1: Add dataset_summary() - [Name]'

Narrative:

This dataset contains 20 sales orders from Q1 2024 (January–March), tracking transactions across four geographic regions (North, South, East, West) with five products (Laptop, Headset, Monitor, Keyboard, Webcam) sold by four sales representatives. The data includes order date, quantity, unit price, and calculated revenue for each transaction, with order values ranging from under $400 to over $12,000. Our quality check confirmed clean data with zero missing values, no duplicate orders, and no negative quantities, ensuring the dataset is reliable for analysis. This foundation allows us to confidently examine regional performance, product trends, and sales rep effectiveness across the quarter.

In [ ]:
# Section 2, Cell 1: Revenue by product

def revenue_by_product(dataframe):
    """Calculate total revenue for each product.

    Args:
        dataframe: sales DataFrame with 'Product' and 'Revenue' columns

    Returns:
        dict mapping product name to total revenue (float), sorted descending
        e.g. {'Laptop': 8400.00, 'Monitor': 1996.00, ...}

    >>> import pandas as pd
    >>> test = pd.DataFrame({'Product': ['A','B','A'], 'Revenue': [100.0, 50.0, 200.0]})
    >>> revenue_by_product(test)
    {'A': 300.0, 'B': 50.0}
    """
    revenue_dict = dataframe.groupby('Product')['Revenue'].sum().to_dict()
    return dict(sorted(revenue_dict.items(), key=lambda x: x[1], reverse=True))


by_product = revenue_by_product(df)
print('Revenue by product:')
for product, rev in by_product.items():
    print(f'  {product}: ${rev:,.2f}')

# Clear outputs -> Commit: 'Section 2: Add revenue_by_product() - [Name]'

In [ ]:
# Section 2, Cell 2: Revenue by region

def revenue_by_region(dataframe):
    """Calculate total revenue for each region.

    Args:
        dataframe: sales DataFrame with 'Region' and 'Revenue' columns

    Returns:
        dict mapping region name to total revenue (float), sorted descending

    >>> import pandas as pd
    >>> test = pd.DataFrame({'Region': ['North','South','North'], 'Revenue': [500.0, 200.0, 300.0]})
    >>> revenue_by_region(test)
    {'North': 800.0, 'South': 200.0}
    """
    revenue_dict = dataframe.groupby('Region')['Revenue'].sum().to_dict()
    return dict(sorted(revenue_dict.items(), key=lambda x: x[1], reverse=True))


by_region = revenue_by_region(df)
print('Revenue by region:')
for region, rev in by_region.items():
    print(f'  {region}: ${rev:,.2f}')

# Clear outputs -> Commit: 'Section 2: Add revenue_by_region() - [Name]'

In [ ]:
# Section 2, Cell 3: Top product by revenue

def top_revenue_item(revenue_dict):
    """Return the name and revenue of the highest-earning item.

    Args:
        revenue_dict: dict mapping name to revenue (output of revenue_by_product
                      or revenue_by_region)

    Returns:
        tuple: (name, revenue)

    >>> top_revenue_item({'Laptop': 8400.0, 'Monitor': 1996.0, 'Headset': 719.92})
    ('Laptop', 8400.0)
    """
    if not revenue_dict:
        return None, 0.0
    top_item = max(revenue_dict.items(), key=lambda x: x[1])
    return top_item


top_product = top_revenue_item(revenue_by_product(df))
top_region  = top_revenue_item(revenue_by_region(df))
print(f'Top product: {top_product[0]} (${top_product[1]:,.2f})')
print(f'Top region:  {top_region[0]} (${top_region[1]:,.2f})')

# Clear outputs -> Commit: 'Section 2: Add top_revenue_item() - [Name]'

Laptops earn the most revenue out of all of the products. The north produces the most revenue out of all of the regions. The sales manager should focus on regions outside the North, since sales are already strong there. 


In [ ]:
# Section 3, Cell 1: Revenue by month

def revenue_by_month(dataframe):
    """Calculate total revenue for each calendar month.

    Args:
        dataframe: sales DataFrame with 'Date' (datetime) and 'Revenue' columns

    Returns:
        dict mapping month name to total revenue, ordered chronologically
        e.g. {'January': 4620.93, 'February': 3639.84, 'March': 11099.65}

    >>> import pandas as pd
    >>> test = pd.DataFrame({
    ...     'Date': pd.to_datetime(['2024-01-05','2024-01-12','2024-02-03']),
    ...     'Revenue': [2400.0, 899.95, 799.90]})
    >>> result = revenue_by_month(test)
    >>> result['January']
    3299.95
    >>> list(result.keys())
    ['January', 'February']
    """
    if dataframe.empty:
        return {}

    # use month_name for labels but keep month numbers for ordering
    df2 = dataframe.copy()
    df2['month_num'] = df2['Date'].dt.month
    df2['month_name'] = df2['Date'].dt.month_name()

    grouped = (
        df2.groupby(['month_num', 'month_name'])['Revenue']
           .sum()
           .reset_index()
           .sort_values('month_num')
    )

    result = {row.month_name: row.Revenue for row in grouped.itertuples()}
    return result



monthly = revenue_by_month(df)
print('Monthly revenue:')
for month, rev in monthly.items():
    print(f'  {month}: ${rev:,.2f}')

# Clear outputs -> Commit: 'Section 3: Add revenue_by_month() - [Name]'}

In [ ]:
# Section 3, Cell 2: Order count by month

def orders_by_month(dataframe):
    """Count the number of orders placed in each calendar month.

    Args:
        dataframe: sales DataFrame with 'Date' (datetime) and 'OrderID' columns

    Returns:
        dict mapping month name to order count, ordered chronologically

    >>> import pandas as pd
    >>> test = pd.DataFrame({
    ...     'Date': pd.to_datetime(['2024-01-05','2024-01-12','2024-02-03']),
    ...     'OrderID': [1001, 1002, 1003]})
    >>> orders_by_month(test)
    {'January': 2, 'February': 1}
    """
    if dataframe.empty:
        return {}

    df2 = dataframe.copy()
    df2['month_num'] = df2['Date'].dt.month
    df2['month_name'] = df2['Date'].dt.month_name()

    grouped = (
        df2.groupby(['month_num', 'month_name'])['OrderID']
           .count()
           .reset_index()
           .sort_values('month_num')
    )

    result = {row.month_name: row.OrderID for row in grouped.itertuples()}
    return result


monthly_orders = orders_by_month(df)
print('Orders per month:')
for month, count in monthly_orders.items():
    print(f'  {month}: {count} orders')

# Clear outputs -> Commit: 'Section 3: Add orders_by_month() - [Name]'

In [ ]:
# Section 3, Cell 3: Best and worst month

def month_comparison(monthly_revenue):
    """Identify the best and worst revenue months.

    Args:
        monthly_revenue: dict from revenue_by_month()

    Returns:
        dict with keys:
          'best_month': str, name of highest-revenue month
          'best_revenue': float
          'worst_month': str, name of lowest-revenue month
          'worst_revenue': float
          'growth_pct': float, percentage change from first to last month

    >>> result = month_comparison({'January': 4000.0, 'February': 3000.0, 'March': 9000.0})
    >>> result['best_month']
    'March'
    >>> result['growth_pct']
    125.0
    """
    if not monthly_revenue:
        return {
            'best_month': None,
            'best_revenue': 0.0,
            'worst_month': None,
            'worst_revenue': 0.0,
            'growth_pct': 0.0
        }

    best_month = max(monthly_revenue.items(), key=lambda x: x[1])
    worst_month = min(monthly_revenue.items(), key=lambda x: x[1])

    # Calculate growth from first to last month
    months = list(monthly_revenue.keys())
    first_month = months[0]
    last_month = months[-1]
    first_revenue = monthly_revenue[first_month]
    last_revenue = monthly_revenue[last_month]

    growth_pct = ((last_revenue - first_revenue) / first_revenue * 100) if first_revenue else 0.0

    return {
        'best_month': best_month[0],
        'best_revenue': best_month[1],
        'worst_month': worst_month[0],
        'worst_revenue': worst_month[1],
        'growth_pct': growth_pct
    }


comparison = month_comparison(monthly)
print(f"Best month:  {comparison['best_month']} (${comparison['best_revenue']:,.2f})")
print(f"Worst month: {comparison['worst_month']} (${comparison['worst_revenue']:,.2f})")
print(f"Q1 growth:   {comparison['growth_pct']:.1f}% from Jan to Mar")

# Clear outputs -> Commit: 'Section 3: Add month_comparison() - [Name]'

Sales across Q1 grew by about 61.4%. The strongest month was March because they made the most revenue during this month. I notice that the more orders the company had the more revenue they made.

In [ ]:
# Section 4, Cell 1: Revenue per sales rep

def revenue_by_rep(dataframe):
    """Calculate total revenue attributed to each sales representative.

    Args:
        dataframe: sales DataFrame with 'SalesRep' and 'Revenue' columns

    Returns:
        dict mapping rep name to total revenue, sorted descending

    >>> import pandas as pd
    >>> test = pd.DataFrame({'SalesRep': ['Alice','Bob','Alice'], 'Revenue': [300.0, 200.0, 150.0]})
    >>> revenue_by_rep(test)
    {'Alice': 450.0, 'Bob': 200.0}
    """
    revenue_dict = dataframe.groupby('SalesRep')['Revenue'].sum().to_dict()
    return dict(sorted(revenue_dict.items(), key=lambda x: x[1], reverse=True))

rep_revenue = revenue_by_rep(df)
print('Revenue by sales rep:')
for rep, rev in rep_revenue.items():
    print(f'  {rep}: ${rev:,.2f}')

# Clear outputs -> Commit: 'Section 4: Add revenue_by_rep() - [Name]'

In [ ]:
# Section 4, Cell 2: Orders per sales rep

def orders_by_rep(dataframe):
    """Count the number of orders handled by each sales rep.

    Args:
        dataframe: sales DataFrame with 'SalesRep' and 'OrderID' columns

    Returns:
        dict mapping rep name to order count, sorted descending

    >>> import pandas as pd
    >>> test = pd.DataFrame({'SalesRep': ['Alice','Bob','Alice'], 'OrderID': [1,2,3]})
    >>> orders_by_rep(test)
    {'Alice': 2, 'Bob': 1}
    """
    orders_dict = dataframe.groupby('SalesRep')['OrderID'].count().to_dict()
    return dict(sorted(orders_dict.items(), key=lambda x: x[1], reverse=True))


rep_orders = orders_by_rep(df)
print('Orders per rep:')
for rep, n in rep_orders.items():
    print(f'  {rep}: {n} orders')

# Clear outputs -> Commit: 'Section 4: Add orders_by_rep() - [Name]'

In [ ]:
# Section 4, Cell 3: Average order value per rep

def avg_order_value_by_rep(dataframe):
    """Calculate the average revenue per order for each sales rep.

    Args:
        dataframe: sales DataFrame with SalesRep, Revenue, OrderID columns

    Returns:
        dict mapping rep name to their average order value (float), sorted descending

    >>> import pandas as pd
    >>> test = pd.DataFrame({'SalesRep': ['Alice','Alice','Bob'],
    ...                      'Revenue': [400.0, 200.0, 300.0],
    ...                      'OrderID': [1,2,3]})
    >>> result = avg_order_value_by_rep(test)
    >>> result['Alice']
    300.0
    >>> result['Bob']
    300.0
    """
    avg_dict = dataframe.groupby('SalesRep')['Revenue'].mean().to_dict()
    return dict(sorted(avg_dict.items(), key=lambda x: x[1], reverse=True))


avg_values = avg_order_value_by_rep(df)
print('Average order value by rep:')
for rep, avg in avg_values.items():
    print(f'  {rep}: ${avg:,.2f} per order')

# Clear outputs -> Commit: 'Section 4: Add avg_order_value_by_rep() - [Name]'

Jordan has the highest sale cost on average per representative. They all handeled the same amount of orders, and jordan had the highest revenue per rep. 

In [ ]:
# Final Integration Check — run after all sections are merged to main
# Repo Owner: run this cell, then clear outputs, commit, and push

print('=' * 55)
print('WEEK 6 ASSIGNMENT: FINAL INTEGRATION CHECK')
print('=' * 55)

all_passed = True

# Check Section 1 functions
try:
    q = check_data_quality(df)
    s = dataset_summary(df)
    print('Section 1: check_data_quality()  OK')
    print('Section 1: dataset_summary()     OK')
except Exception as e:
    print(f'Section 1 ERROR: {e}')
    all_passed = False

# Check Section 2 functions
try:
    bp = revenue_by_product(df)
    br = revenue_by_region(df)
    tp = top_revenue_item(bp)
    print('Section 2: revenue_by_product()  OK')
    print('Section 2: revenue_by_region()   OK')
    print('Section 2: top_revenue_item()    OK')
except Exception as e:
    print(f'Section 2 ERROR: {e}')
    all_passed = False

# Check Section 3 functions
try:
    mo = revenue_by_month(df)
    oc = orders_by_month(df)
    mc = month_comparison(mo)
    print('Section 3: revenue_by_month()    OK')
    print('Section 3: orders_by_month()     OK')
    print('Section 3: month_comparison()    OK')
except Exception as e:
    print(f'Section 3 ERROR: {e}')
    all_passed = False

# Check Section 4 (if 4-person team)
try:
    rr = revenue_by_rep(df)
    ro = orders_by_rep(df)
    av = avg_order_value_by_rep(df)
    print('Section 4: revenue_by_rep()      OK')
    print('Section 4: orders_by_rep()       OK')
    print('Section 4: avg_order_value_by_rep() OK')
except NameError:
    print('Section 4: (3-person team, skipped)')
except Exception as e:
    print(f'Section 4 ERROR: {e}')
    all_passed = False

print()
print('=' * 55)
if all_passed:
    print('ALL FUNCTIONS PASS — ready to submit!')
else:
    print('SOME FUNCTIONS FAILED — fix before submitting')
print('=' * 55)

# Clear outputs -> Commit: 'Final integration check passes - [Team Name]' -> Push